# `ModelSpec`, the log density, and the jax interpreter

A backend fits a *model spec*, not a Python callable (plan review A2): the mean expression, the
outcome column, a likelihood, and a prior for every parameter. The same tree is evaluated by
numpy (`value`, `log_density`) and by jax (`compile_jax`, `compile_log_density`), and the two
must agree — that agreement is gate 9, the numerical replacement for "both call `forward()`".

Three nodes support panel models: `Reduce` (sum/mean/max along the last axis, used to
normalize carryover weights), `Gather` (a vector parameter indexed by an integer column — a
unit-level intercept), and a vector `Const` (a lag index).

In [ ]:
import numpy as np

from axiom.core import (
    PRIOR_HYPER, Add, Const, Convolve, D, Data, DesignMatrix, Div, Gather, Likelihood, LikelihoodFamily,
    ModelSpec, Mul, Param, Pow, Prior, PriorFamily, Reduce, Spec, compile_jax, compile_log_density,
    constrain, dimension, dimensionless, free_parameters, jax_available, log_density, log_prior,
    require_jax, unconstrain, value,
)

## Priors and hierarchy

`PRIOR_HYPER` lists the hyperparameters each family takes. A hyperparameter may *name another
parameter* — that is how a hierarchy is declared.

In [ ]:
print(PRIOR_HYPER)
fam: PriorFamily = "halfnormal"
a_mean = Param(name="a_mean", dimension=D.outcome, prior=Prior(family="normal", hyper={"mu": 0.0, "sigma": 10.0}))
a_sd = Param(name="a_sd", dimension=D.outcome, prior=Prior(family=fam, hyper={"sigma": 5.0}))
alpha = Param(name="alpha", dimension=D.outcome, shape=(3,), prior=Prior(family="normal", hyper={"mu": "a_mean", "sigma": "a_sd"}))
print(alpha.prior.parents, alpha.size)

## Carryover weights as a pure expression

No opaque function is needed: `Pow(lam, lags) / Reduce(sum)` is the normalized geometric
weight vector, and `Convolve` applies it causally. The jax interpreter therefore covers it.

In [ ]:
lam = Param(name="lam", dimension=dimensionless(), prior=Prior(family="beta", hyper={"alpha": 2.0, "beta": 2.0}))
lags = Const(value=tuple(float(i) for i in range(6)), dimension=dimensionless())
raw = Pow(base=lam, exponent=lags)
weights = Div(numerator=raw, denominator=Reduce(op="sum", arg=raw))
dose = Data(name="dose", dimension=D.currency)
carried = Convolve(signal=dose, kernel=weights)
impulse = np.array([100.0, 0, 0, 0, 0, 0, 0, 0])
print(dimension(carried), value(carried, data={"dose": impulse}, params={"lam": 0.5}).round(3))

## A hierarchical panel model

In [ ]:
unit = Data(name="unit", dimension=dimensionless())
y = Data(name="y", dimension=D.outcome)
k = Param(name="k", dimension=D.currency, prior=Prior(family="lognormal", hyper={"mu": float(np.log(50)), "sigma": 0.5}))
s = Param(name="s", dimension=dimensionless(), prior=Prior(family="gamma", hyper={"alpha": 4.0, "beta": 2.0}))
beta = Param(name="beta", dimension=D.outcome, prior=Prior(family="halfnormal", hyper={"sigma": 20.0}))
sigma = Param(name="sigma", dimension=D.outcome, prior=Prior(family="halfnormal", hyper={"sigma": 5.0}))
u = Div(numerator=carried, denominator=k)
hill = Mul(factors=(beta, Div(numerator=Pow(base=u, exponent=s), denominator=Add(terms=(Const(value=1.0, dimension=dimensionless()), Pow(base=u, exponent=s))))))
mean = Add(terms=(Gather(source=alpha, index=unit), hill))

family: LikelihoodFamily = "normal"
model = ModelSpec(
    name="panel_hill",
    mean=mean,
    outcome=y,
    likelihood=Likelihood(family=family, scale="sigma"),
    parameters=(k, s, beta, lam, a_mean, a_sd, alpha, sigma),
)
print([p.name for p in free_parameters(model)])
print("shapes:", [p.name for p in model.shapes], "| scales:", [p.name for p in model.scales])
print(Spec.from_json(model.to_json()) == model)

## Unconstrained coordinates and the log density

Inference happens on $\mathbb{R}^k$: `unconstrain` maps each parameter by its support (log for
positive, logit for the unit interval), `constrain` maps back and returns the log-Jacobian, and
`log_density` = log prior + Jacobian + log likelihood.

In [ ]:
rng = np.random.default_rng(0)
n = 24
data = {"unit": rng.integers(0, 3, n), "dose": rng.uniform(0, 150, n)}
theta = {"k": 50.0, "s": 2.0, "beta": 10.0, "lam": 0.5, "a_mean": 1.0, "a_sd": 0.5, "alpha": np.array([0.5, 1.0, 1.5]), "sigma": 1.0}
data["y"] = value(mean, data=data, params=theta) + rng.normal(0, 1, n)

z = unconstrain(model, theta)
back, log_jac = constrain(model, z)
print({k_: np.round(v, 3) for k_, v in z.items() if k_ in ("k", "lam", "s")}, "| log|J| =", round(log_jac, 3))
print("round-trips:", all(np.allclose(back[k_], theta[k_]) for k_ in theta))
print("log prior:", round(log_prior(model, theta), 3), "| log density:", round(log_density(model, data, z), 3))

## The jax interpreter

`compile_jax` turns the tree into a traceable function; `compile_log_density` does the same for
the whole posterior. When jax is not installed, `require_jax()` returns a typed `Unsupported`
instead of an `ImportError`.

In [ ]:
print("jax available:", jax_available(), "|", require_jax())
if jax_available():
    import jax

    jax.config.update("jax_enable_x64", True)
    f_mean = compile_jax(mean)
    f_ld = compile_log_density(model)
    print("mean agrees:", np.allclose(np.asarray(f_mean(data, theta)), value(mean, data=data, params=theta)))
    print("log density agrees:", abs(float(f_ld(data, z)) - log_density(model, data, z)) < 1e-8)
    grad = jax.grad(lambda zz: f_ld(data, zz))(z)
    print({k_: np.asarray(v).shape for k_, v in grad.items()})

## `DesignMatrix`

`SupportsForward.linearize` returns a `DesignMatrix`: at a fixed point of the nonlinear
parameters, the mean is `offset + X @ theta[columns]`. Its invariant against `forward()` is
gate 9's other half and lands with `surface.linearize`.

In [ ]:
X = np.column_stack([np.ones(4), np.arange(4.0)])
dm = DesignMatrix(X=X, columns=("a", "b"), offset=np.zeros(4), at={"k": np.array(50.0)})
print(dm.predict({"a": 1.0, "b": 2.0}))